## Contexto
Este estudo tem como objetivo construir um modelo de **regressão logística binária** para prever a probabilidade de inadimplência em solicitações de crédito. Esse tipo de modelo é bastante utilizado no setor financeiro por combinar boa capacidade preditiva com transparência nos resultados, permitindo identificar o impacto de cada variável no risco de crédito.

In [56]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [39]:
credit_card_atributes = pd.read_csv('../data/application_record.csv')
credit_card_status = pd.read_csv('../data/credit_record.csv')


In [40]:
# Verificando dimensões dos DataFrames
print(credit_card_atributes.shape)
print(credit_card_status.shape)

(438557, 18)
(1048575, 3)


In [43]:
# junção do Dataframes
df_credit_card = pd.merge(credit_card_atributes, credit_card_status, on='ID', how='left')

In [44]:
df_credit_card.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MONTHS_BALANCE,STATUS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,0.0,C
1,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-1.0,C
2,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-2.0,C
3,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-3.0,C
4,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,-4.0,C


In [45]:
df_credit_card.columns

Index(['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN',
       'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE',
       'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'MONTHS_BALANCE',
       'STATUS'],
      dtype='object')

# Dicionário de dados
| Variável   | Tradução           | Descrição                                |
|------------|--------------------|------------------------------------------|
| ID         | ID DO CLIENTE      | NÚMERO DE IDENTIFICAÇÃO DO CLIENTE       |
| CODE_GENDER| GÊNERO             | GÊNERO DO CLIENTE                        |
| FLAG_OWN_CAR   | TEM UM CARRO  ?         | Y - POSSUI, N - NÃO POSSUI      |
| FLAG_OWN_REALTY  | TEM CASA ?          | Y - POSSUI, N - NÃO POSSUI        |
| CNT_CHILDREN| NÚMERO DE FILHOS          | QTD DE FILHOS                    |
| AMT_INCOME_TOTAL| RENDA ANUAL| RENDA ANUAL EM UNIDADES MONETÁRIAS DO CLIENTE |
| NAME_INCOME_TYPE| CATEGORIA DE RENDA | TIPO DE OCUPAÇÃO           |
| NAME_EDUCATION_TYPE| TIPO DE EDUÇÃO | NÍVEL DE FORMAÇÃO EDUCACIONAL DO CLIENTE           |
| NAME_FAMILY_STATUS | STATUS CIVIL  | STATUS CIVIL DO CLIENTE       |
| NAME_HOUSING_TYPE | TIPO DE MORADIA  | TIPO DE MORADIA DO CLIENTE(APARTAMENTO ALUGADO, CASA, COM OS PAIS)     |
| DAYS_BIRTH | ANIVERSÁRIO | CONTAGEM REGRESSIVA A PARTIR DO DIA ATUAL (0), -1 SIGNIFICA ONTEM     |
| DAYS_EMPLOYED | DIAS EMPREGADO | QUANTIDADE DE DIAS QUE O CLIENTE ESTÁ EMPREGADO SE POSITIVO SIGNIFICA QUE O CLIENTE ESTÁ DESEMPREGADA |
| FLAG_MOBIL | TELEFONE CELULAR | POSSUI TELEFONE CELULAR |
| FLAG_WORK_PHONE | TELEFONE CELULAR CORPORATIVO | POSSUI TELEFONE CELULAR CORPORATIVO |
| FLAG_WORK_PHONE | TELEFONE FIXO | POSSUI TELEFONE FIXO |
| FLAG_EMAIL | E-MAIL| POSSUI UM E-MAIL|
| OCCUPATION_TYPE | PROFISSÃO | PROFISSÃO DO CLIENTE|
| CNT_FAM_MEMBERS| MEMBROS DA FAMÍLIA | QUANTIDADE DE MEMBROS NA FAMÍLIA|
| MONTHS_BALANCE| MES DE REGISTRO | MÊS EM QUE O REGISTRO FOI FEITO 0 MÊS ATUAL -1 UM MÊS ANTES E ETC.|
| STATUS | STATUS DO CRÉDITO | INDICA A SITUAÇÃO DO CLIENTE EM DIAS (0 - 1 A 29, 1 - 30 A 50, 2 - 60 A 89, 3 - 90 A 119, 4 - 120 A 149, 5 - MAIS DE 150 OU DÍVIDA INCOBRÁVEL, C- DIVIDA QUITADA NO MES , X - NENHUM EMPRESTIMO NO MES).|






In [46]:
df_credit_card.shape

(1179815, 20)

In [47]:
df_credit_card.dtypes

ID                       int64
CODE_GENDER             object
FLAG_OWN_CAR            object
FLAG_OWN_REALTY         object
CNT_CHILDREN             int64
AMT_INCOME_TOTAL       float64
NAME_INCOME_TYPE        object
NAME_EDUCATION_TYPE     object
NAME_FAMILY_STATUS      object
NAME_HOUSING_TYPE       object
DAYS_BIRTH               int64
DAYS_EMPLOYED            int64
FLAG_MOBIL               int64
FLAG_WORK_PHONE          int64
FLAG_PHONE               int64
FLAG_EMAIL               int64
OCCUPATION_TYPE         object
CNT_FAM_MEMBERS        float64
MONTHS_BALANCE         float64
STATUS                  object
dtype: object

In [48]:
df_credit_card.isnull().sum()/len(df_credit_card)

ID                     0.000000
CODE_GENDER            0.000000
FLAG_OWN_CAR           0.000000
FLAG_OWN_REALTY        0.000000
CNT_CHILDREN           0.000000
AMT_INCOME_TOTAL       0.000000
NAME_INCOME_TYPE       0.000000
NAME_EDUCATION_TYPE    0.000000
NAME_FAMILY_STATUS     0.000000
NAME_HOUSING_TYPE      0.000000
DAYS_BIRTH             0.000000
DAYS_EMPLOYED          0.000000
FLAG_MOBIL             0.000000
FLAG_WORK_PHONE        0.000000
FLAG_PHONE             0.000000
FLAG_EMAIL             0.000000
OCCUPATION_TYPE        0.307614
CNT_FAM_MEMBERS        0.000000
MONTHS_BALANCE         0.340816
STATUS                 0.340816
dtype: float64

In [49]:
# removendo dados nulos da coluana alvo 'STATUS'
df_credit_card = df_credit_card.dropna(subset=['STATUS'])

In [52]:
# Valores nulos após remoção na coluna 'STATUS
df_credit_card.isnull().sum()/len(df_credit_card)

ID                     0.000000
CODE_GENDER            0.000000
FLAG_OWN_CAR           0.000000
FLAG_OWN_REALTY        0.000000
CNT_CHILDREN           0.000000
AMT_INCOME_TOTAL       0.000000
NAME_INCOME_TYPE       0.000000
NAME_EDUCATION_TYPE    0.000000
NAME_FAMILY_STATUS     0.000000
NAME_HOUSING_TYPE      0.000000
DAYS_BIRTH             0.000000
DAYS_EMPLOYED          0.000000
FLAG_MOBIL             0.000000
FLAG_WORK_PHONE        0.000000
FLAG_PHONE             0.000000
FLAG_EMAIL             0.000000
OCCUPATION_TYPE        0.308658
CNT_FAM_MEMBERS        0.000000
MONTHS_BALANCE         0.000000
STATUS                 0.000000
dtype: float64

OCCUPATION - PROFISSSÃO FOI A ÚNICA VARIÁVEL QUE APÓS TRATAMENTO DA COLUNA ALVO STATUS, OS VALORES NULOS DESTA COLUNA SERÃO REFERENCIADOS COMO INDEFINIDO VISTO QUE É UMA COLUNA CATEGÓRICA.

In [53]:
df_credit_card['OCCUPATION_TYPE'].unique()

array([nan, 'Security staff', 'Sales staff', 'Accountants', 'Laborers',
       'Managers', 'Drivers', 'Core staff', 'High skill tech staff',
       'Cleaning staff', 'Private service staff', 'Cooking staff',
       'Low-skill Laborers', 'Medicine staff', 'Secretaries',
       'Waiters/barmen staff', 'HR staff', 'Realty agents', 'IT staff'],
      dtype=object)

In [57]:
df_credit_card['OCCUPATION_TYPE'].fillna('UNDEFINED', inplace=True)

In [60]:
# Tamanho final do Dataframe
df_credit_card.shape

(777715, 20)